<a href="https://colab.research.google.com/github/ubsuny/PHY386/blob/Homework2026/2026/FinalProject/Final_05_GaiaVariables.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project — Topic 5: Gaia Variable-Star Classification (42 pts)

## Learning Outcomes
- Compose an ADQL join across two Gaia DR3 tables using `astroquery.gaia`.
- Handle severe class imbalance in a multi-class classification problem.
- Train the four HW5/HW6 classifiers (Decision Tree, k-Nearest Neighbors, Random Forest, Multi-Layer Perceptron) on real stellar data and compare their strengths.
- Interpret a multi-class confusion matrix in terms of the physics of stellar variability.
- Overlay the Leavitt period–luminosity relation on your predictions to test whether the classifier recovers textbook stellar physics.

## GitHub Workflow
Fork → `yourname-final` → PR into `Homework2026`. Reviewer `@laserlab`, milestone `Final-2026`.

## Heads-up on the Gaia archive
The Gaia archive occasionally reports transient instability — the ADQL server may prompt "the archive is unstable and we are working to resolve this issue." If your query hangs, wait 5 minutes and retry, or reduce `SELECT TOP 20000` to a smaller number.

## Background: Why Stars Vary

Most stars are boringly steady. The ones that aren't come in several physically distinct flavors — and those flavors are the cornerstone of modern astrophysics.

- **Pulsating variables** (RR Lyrae, Cepheids, δ Scuti, long-period variables / Miras) — the whole star expands and contracts. For Cepheids and RR Lyrae, the pulsation period is tightly tied to luminosity (the **Leavitt / period–luminosity relation**), which makes them "standard candles" in cosmology.
- **Eclipsing binaries** — two stars in mutual orbit; the light curve dips when one blocks the other. Periods range from hours to years.
- **Rotational variables** — spotted stars modulated by rotation.
- **Cataclysmic variables / eruptive** — novae, dwarf novae: violent brightness changes.

Gaia DR3's `gaiadr3.vari_classifier_result` table assigns each source a `best_class_name` label drawn from this zoo, plus a confidence score. The task: recover those labels from only the mean photometry and astrometry — no light curves.

Why is that interesting? Because the labels encode different *physics* (pulsation vs. eclipsing geometry vs. rotation), and those different physics imprint themselves on the stars' color, luminosity, and distance. If a classifier can separate RR Lyrae from Cepheids using just position in the H–R diagram, it's effectively rediscovering the instability strip.

In [ ]:
# Required packages (uncomment for Colab or fresh environment):
# %pip install astroquery scikit-learn pandas --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 11, 'axes.titlesize': 11,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
})

## Worked Example: ADQL Join
The query below joins the variability-classifier table with `gaia_source` on `source_id` and pulls the features we'll classify on. We restrict to `best_class_score > 0.9` so the labels are reliable, and cap at 20,000 rows to stay polite to the server.

In [ ]:
from astroquery.gaia import Gaia

QUERY = """
SELECT TOP 20000
  v.source_id, v.best_class_name, v.best_class_score,
  g.phot_g_mean_mag, g.bp_rp, g.parallax, g.pmra, g.pmdec,
  g.phot_variable_flag
FROM gaiadr3.vari_classifier_result AS v
JOIN gaiadr3.gaia_source AS g USING (source_id)
WHERE v.best_class_score > 0.9
  AND g.parallax IS NOT NULL
  AND g.bp_rp IS NOT NULL
"""

job = Gaia.launch_job_async(QUERY)
df = job.get_results().to_pandas()
print(f"{len(df)} sources")
print(df['best_class_name'].value_counts().head(10))
df.head()

## Part 1 — Class Balance and Feature Engineering (10 pts)

### Task 1.1 — Pick your classes (3 pts)
The variability classifier has many classes, very unevenly populated. Restrict to the top N most common — 4 to 7 is sensible — and drop the rest. Report how many stars per class you keep.

### Task 1.2 — Add an absolute-magnitude feature (4 pts)
Compute:
$$M_G = G + 5 \log_{10}(\varpi_\text{mas}) - 10$$
where $G$ is `phot_g_mean_mag` and $\varpi_\text{mas}$ is `parallax` in milliarcseconds. Drop rows with non-positive parallax. This is your most physically informative feature — it places each star on the H–R diagram.

### Task 1.3 — Feature list (3 pts)
Final feature vector per star: `[bp_rp, abs_mag, pmra, pmdec]`. Label: `best_class_name`. Any additional features you want to try (e.g. total proper motion $\sqrt{\mu_\alpha^2 + \mu_\delta^2}$, galactic latitude) — justify briefly.

In [ ]:
# Part 1: your code here

## Part 2 — Train the Four Classifiers (14 pts)

Use the full HW5 + HW6 lineup: Decision Tree, k-Nearest Neighbors, Random Forest, Multi-Layer Perceptron.

### Task 2.1 — Pipeline (3 pts)
Train/test split stratified on the label. Standardize features with `StandardScaler`. Address class imbalance either by downsampling large classes or by passing `class_weight='balanced'` (where supported). State which you chose and why.

### Task 2.2 — Train and evaluate (8 pts, 2 per classifier)
For each of the four, report test-set accuracy **and** macro-F1 (which penalizes ignoring rare classes). Plot the confusion matrix.

### Task 2.3 — Summary table (3 pts)

| Classifier | Accuracy | Macro-F1 | Notes |
|------------|----------|----------|-------|
| Decision Tree | ... | ... | ... |
| k-Nearest Neighbors | ... | ... | ... |
| Random Forest | ... | ... | ... |
| Multi-Layer Perceptron | ... | ... | ... |

In [ ]:
# Part 2: your code here
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.neural_network import MLPClassifier
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

## Part 3 — Physics-in-the-H–R-Diagram (10 pts)

### Task 3.1 — Colored H–R diagram (4 pts)
Plot $M_G$ (y, inverted) vs. $B_P - R_P$ (x), colored by **true** class. Does each variability class occupy a recognizable region? RR Lyrae should land near the instability strip; Miras in the upper-right red-giant corner; eclipsing binaries spread across the main sequence.

### Task 3.2 — Classifier decision regions (4 pts)
Take a 2D slice (fix proper motions at their median values) and plot the Random Forest's decision regions in ($B_P - R_P$, $M_G$). Do the regions match the physical zones from Task 3.1?

### Task 3.3 — Physics discussion (2 pts)
In 4–6 sentences: which classes are most-confused-with-which in the confusion matrix, and does that match where they overlap on the H–R diagram?

In [ ]:
# Part 3: your code here

## Part 4 — Stretch: The Leavitt Law (8 pts)

If you kept Cepheids and/or RR Lyrae in your class set, you can test whether the classifier has implicitly learned the **period–luminosity relation** — the Leavitt law that made Cepheids the standard candle underpinning modern distance measurements.

The challenge: periods are in a separate Gaia table (`gaiadr3.vari_cepheid`, `gaiadr3.vari_rrlyrae`). You need a second ADQL query to pull them.

### Task 4.1 — Query Cepheid or RR Lyrae periods (4 pts)
```sql
SELECT source_id, pf, p1_o
FROM gaiadr3.vari_cepheid
```
Join on `source_id` with your main DataFrame.

### Task 4.2 — Period–luminosity plot (4 pts)
Plot $M_G$ vs. $\log_{10}(P / \mathrm{day})$ for Cepheids (and/or RR Lyrae). Fit a line. Compare your slope and intercept to the published Leavitt relation. Color points by classifier-predicted class — do misclassified stars fall off the relation?

In [ ]:
# Part 4 (optional stretch): your code here

## Submission
**Good commit** ✅: `Add multi-class confusion matrix and HR-diagram decision regions`

**Bad commit** ❌: `fix`, `wip`

### Checklist
- [ ] Notebook in `2026/FinalProject/<yourname>/`
- [ ] Runs top-to-bottom on a fresh kernel
- [ ] Type annotations and NumPy docstrings on your functions
- [ ] All plots labeled (HR diagram with inverted y-axis!)
- [ ] Tasks sum to ~42 points
- [ ] PR into `Homework2026`, reviewer `@laserlab`, milestone `Final-2026`
- [ ] AI usage disclosed

## Resources
- [`astroquery.gaia`](https://astroquery.readthedocs.io/en/latest/gaia/gaia.html)
- [Gaia DR3 variability tables](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_variability_tables/)
- [`vari_classifier_result` column dictionary](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_variability_tables/ssec_dm_vari_classifier_result.html)
- Leavitt, H. S. & Pickering, E. C. (1912). *Periods of 25 variable stars in the Small Magellanic Cloud.* Harvard College Observatory Circular 173.
- Clementini, G. et al. (2023). *Gaia Data Release 3: Specific processing and validation of all-sky RR Lyrae and Cepheid stars.* A&A 674, A18.